In [1]:
!pip uninstall opencv-python -y
!pip install opencv-python-headless==4.5.4.60
!pip install torch torchvision albumentations matplotlib craft-text-detector

Found existing installation: opencv-python 4.11.0.86
Uninstalling opencv-python-4.11.0.86:
  Successfully uninstalled opencv-python-4.11.0.86
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 MB 9.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 MB 8.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  no

In [3]:
!pip install albumentations opencv-python

In [5]:
# Path ke gambar dan label
import zipfile
import os

with zipfile.ZipFile("test_dummy_ktp.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

In [6]:
import zipfile
with zipfile.ZipFile('craft_mlt_25k.zip', 'r') as zip_ref:
    zip_ref.extractall('craft_weights')

In [11]:
!git clone https://github.com/clovaai/CRAFT-pytorch
%cd CRAFT-pytorch
!pip install -r requirements.txt

[Errno 2] No such file or directory: 'CRAFT-pytorch'
/content/CRAFT-pytorch
ERROR: Could not find a version that satisfies the requirement torch==0.4.1.post2 (from versions: 1.13.0, 1.13.1, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0)
ERROR: No matching distribution found for torch==0.4.1.post2


In [7]:
# Langkah awal: import library
import os
import cv2
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
from sklearn.metrics import classification_report
from PIL import Image, ImageDraw, ImageFont
from craft_text_detector import Craft
craft = Craft(craft_model_path='craft_weights/craft_mlt_25k.pth', cuda=torch.cuda.is_available())

ModuleNotFoundError: No module named 'craft_text_detector'

In [ ]:
image_dir = "/content/test_dummy_ktp"

In [ ]:
import string

VOCAB = list(string.ascii_uppercase + string.digits + ",./:;- '" + '"')
char_to_idx = {ch: idx+1 for idx, ch in enumerate(VOCAB)}  # 0 reserved for CTC blank
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

In [ ]:
# Definisi model CRNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
crnn_model_path = "/content/best.pth"

class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.rnn = nn.LSTM(32 * 16, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, w, -1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x

model = CRNN(num_classes=len(char_to_idx) + 1).to(device)
model.load_state_dict(torch.load(crnn_model_path, map_location=device))
model.eval()

In [ ]:
# Load detektor teks CRAFT
craft = Craft(output_dir=None, crop_type="poly", cuda=torch.cuda.is_available())

# Fungsi decoding CTC ke teks
def decode_ctc(preds):
    preds = preds.permute(1, 0, 2)
    preds = torch.nn.functional.log_softmax(preds, dim=2)
    pred_texts = []
    for pred in preds:
        pred_idx = pred.argmax(dim=1).cpu().numpy()
        prev_char = None
        decoded = ""
        for idx in pred_idx:
            if idx != 0 and idx != prev_char:
                decoded += idx_to_char.get(idx, '')
            prev_char = idx
        pred_texts.append(decoded)
    return " ".join(pred_texts)

In [ ]:
def ocr_image(img_path, use_gpu=True, save_visualization=True, visualize=True):
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    prediction_result = craft.detect_text(img_path)
    boxes = prediction_result["boxes"]

    transform = transforms.Compose([
        transforms.Resize((32, 100)),
        transforms.Grayscale(),
        transforms.ToTensor(),
    ])

    font = ImageFont.truetype("arial.ttf", 16) if os.name == 'nt' else ImageFont.load_default()
    texts = []

    for box in boxes:
        x_min = int(min(box[:, 0]))
        x_max = int(max(box[:, 0]))
        y_min = int(min(box[:, 1]))
        y_max = int(max(box[:, 1]))
        crop = image.crop((x_min, y_min, x_max, y_max))

        input_tensor = transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            preds = model(input_tensor)
            text = decode_ctc(preds)
        texts.append(text)

        draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
        draw.text((x_min, y_min - 10), text, fill="red", font=font)

    if save_visualization:
        vis_path = img_path.replace(".jpg", "_ocr_result.jpg")
        image.save(vis_path)

    if visualize:
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        plt.axis("off")
        plt.title("Hasil OCR Visual")
        plt.show()

    return texts

In [ ]:
def resize_image(image_path, scale):
    image = cv2.imread(image_path)
    height, width = image.shape[:2]
    new_size = (int(width * scale), int(height * scale))
    resized_image = cv2.resize(image, new_size, interpolation=cv2.INTER_AREA)
    temp_path = image_path.replace(".jpg", f"_resized_{int(scale*100)}.jpg")
    cv2.imwrite(temp_path, resized_image)
    return temp_path

def evaluate_resolution_accuracy():
    scales = {"tinggi": 1.0, "sedang": 0.7, "rendah": 0.4}
    results = {"tinggi": [], "sedang": [], "rendah": []}

    for i in range(1, 11):
        original_path = os.path.join(image_dir, f"ktp_{i}.jpg")
        gt_file = os.path.join(image_dir, f"ktp_{i}.txt")
        with open(gt_file, 'r', encoding='utf-8') as f:
            ground_truth = f.read().strip()

        for label, scale in scales.items():
            resized_path = resize_image(original_path, scale)
            texts = ocr_image(resized_path)
            pred_text = " ".join(texts)
            results[label].append({"gt": ground_truth, "pred": pred_text})

    for res, data in results.items():
        y_true = [item["gt"] for item in data]
        y_pred = [item["pred"] for item in data]
        print(f"\n=== Resolusi: {res.upper()} ===")
        print(classification_report(y_true, y_pred, zero_division=0))

    return results

In [ ]:
def evaluate_all():
    results = {}
    total_time_cpu = 0
    total_time_gpu = 0

    for i in range(1, 11):
        img_file = os.path.join(image_dir, f"ktp_{i}.jpg")
        gt_file = os.path.join(image_dir, f"ktp_{i}.txt")

        with open(gt_file, 'r', encoding='utf-8') as f:
            ground_truth = f.read().strip()

        torch.cuda.is_available = lambda: False
        start = time.time()
        cpu_texts = ocr_image(img_file)
        cpu_duration = time.time() - start
        total_time_cpu += cpu_duration

        torch.cuda.is_available = lambda: True
        start = time.time()
        gpu_texts = ocr_image(img_file)
        gpu_duration = time.time() - start
        total_time_gpu += gpu_duration

        results[f"ktp_{i}"] = {
            "gt": ground_truth,
            "cpu_texts": cpu_texts,
            "gpu_texts": gpu_texts,
            "cpu_time": cpu_duration,
            "gpu_time": gpu_duration
        }

    print(f"\nTotal CPU time: {total_time_cpu:.2f}s")
    print(f"Total GPU time: {total_time_gpu:.2f}s")
    return results

In [ ]:


def ocr_image(img_path, use_gpu=True, save_visualization=True, visualize=True):
    # Buka gambar asli
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    # Deteksi kotak teks menggunakan CRAFT
    prediction_result = craft.detect_text(img_path)
    boxes = prediction_result["boxes"]

    result = []  # Format seperti EasyOCR: [text, confidence, box]

    # Transformasi untuk input ke model OCR
    transform = transforms.Compose([
        transforms.Resize((32, 100)),
        transforms.Grayscale(),
        transforms.ToTensor(),
    ])

    font = ImageFont.truetype("arial.ttf", 16) if os.name == 'nt' else ImageFont.load_default()

    for box in boxes:
        # Ambil koordinat
        x_min = int(min(box[:, 0]))
        x_max = int(max(box[:, 0]))
        y_min = int(min(box[:, 1]))
        y_max = int(max(box[:, 1]))

        # Potong area teks
        cropped = image.crop((x_min, y_min, x_max, y_max))
        input_tensor = transform(cropped).unsqueeze(0).to(device)

        # Prediksi teks
        with torch.no_grad():
            preds = model(input_tensor)
            text = decode_ctc(preds)
            confidence = 1.0  # Jika belum ada confidence score

        result.append([text, confidence, box.tolist()])

        # Gambar kotak dan teks di atas gambar
        draw.polygon([tuple(p) for p in box], outline="red", width=2)
        draw.text((x_min, y_min - 15), text, fill="blue", font=font)

    # Simpan atau tampilkan hasil visualisasi
    if save_visualization:
        vis_path = img_path.replace(".jpg", "_ocr_result.jpg")
        image.save(vis_path)
        print(f"[INFO] Gambar hasil OCR disimpan di: {vis_path}")

    if visualize:
        plt.figure(figsize=(12, 8))
        plt.imshow(image)
        plt.axis("off")
        plt.title("Hasil OCR Visualisasi")
        plt.show()

    return result

In [ ]:
def cleanup():
    craft.unload_craftnet_model()
    craft.unload_refinenet_model()
